In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import pymysql
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备：", device)
conn = pymysql.connect(
    host='127.0.0.1', port=3306,
    user='root', password='root',
    database='guandoupai_db', charset='utf8mb4'
)
# 构造玩家行为特征
df = pd.read_sql("""
SELECT 
    a.session_id,
    a.player_id,
    p.identity,
    COUNT(*) as total_actions,
    SUM(CASE WHEN a.skill='bribe' THEN 1 ELSE 0 END) as bribe_count,
    SUM(CASE WHEN a.skill='assassinate' THEN 1 ELSE 0 END) as assassinate_count,
    SUM(CASE WHEN a.skill='search' THEN 1 ELSE 0 END) as search_count,
    SUM(CASE WHEN a.skill='detain' THEN 1 ELSE 0 END) as detain_count,
    SUM(CASE WHEN a.skill='report' THEN 1 ELSE 0 END) as report_count,
    SUM(CASE WHEN a.skill='peoples_will' THEN 1 ELSE 0 END) as peoples_will_count,
    SUM(CASE WHEN a.is_legal=0 THEN 1 ELSE 0 END) as illegal_count
FROM game_actions a
JOIN game_players p ON a.session_id = p.session_id AND a.player_id = p.player_id
GROUP BY a.session_id, a.player_id, p.identity
""", conn)
conn.close()
# 标签：是否为贪官
df['label'] = (df['identity'] == 'corrupt').astype(int)
# 特征列
feature_cols = [
    'total_actions', 'bribe_count', 'assassinate_count', 
    'search_count', 'detain_count', 'report_count', 
    'peoples_will_count', 'illegal_count'
]
X = df[feature_cols].values
y = df['label'].values
# 标准化
scaler = StandardScaler()
X = scaler.fit_transform(X)
# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)
# 身份识别神经网络
class IdentityPredictor(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)
model = IdentityPredictor().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)
# 训练
epochs = 300
loss_history = []
for epoch in range(epochs):
    model.train()
    pred = model(X_train)
    loss = criterion(pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    
    if (epoch+1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
# 评估
model.eval()
with torch.no_grad():
    test_pred = model(X_test)
    test_pred_label = (test_pred > 0.5).int()
    acc = accuracy_score(y_test.cpu(), test_pred_label.cpu())
    print(f"\n测试集准确率：{acc:.2%}")
    print("\n分类报告：")
    print(classification_report(y_test.cpu(), test_pred_label.cpu(), target_names=["好人", "贪官"]))



使用设备： cpu


C:\Users\杨\AppData\Local\Temp\ipykernel_5052\556468397.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


Epoch [50/300], Loss: 0.4258
Epoch [100/300], Loss: 0.2938
Epoch [150/300], Loss: 0.1870
Epoch [200/300], Loss: 0.1423
Epoch [250/300], Loss: 0.1063
Epoch [300/300], Loss: 0.0922

测试集准确率：85.37%

分类报告：
              precision    recall  f1-score   support

          好人       0.89      0.94      0.91        33
          贪官       0.67      0.50      0.57         8

    accuracy                           0.85        41
   macro avg       0.78      0.72      0.74        41
weighted avg       0.84      0.85      0.85        41

